# Tech Challenge FIAP Fase 4 — Execução Completa no Colab

Este notebook executa a **fusão multimodal** de uma só vez:

1. Baixa os dados reais do eICU Demo (PhysioNet).
2. Usa um vídeo de fisioterapia (padrão do repo ou um específico do REHAB24-6).
3. Roda `main.py`, que orquestra o módulo clínico, o módulo de vídeo e a fusão.
4. Exibe o relatório final consolidado.

> Ambiente de execução: **CPU é suficiente**. GPU não acelera significativamente o MediaPipe.

## 1. Clonar repositório e instalar dependências

Clona o repositório, instala as dependências e posiciona o notebook na raiz do projeto.
Os pacotes internos (`eicu_anomaly_detection`, `modulo_video`, `audio`, `fusion`) são importados diretamente a partir da raiz do repositório, então **não é necessário** executar `pip install -e .`.


In [ ]:
import os

BRANCH = os.environ.get('NOTEBOOK_BRANCH', 'main')
print(f'Clonando/Puxando branch: {BRANCH}')

PROJETO = '/content/Tech-Challenge-FIAP-FASE4'
if not os.path.exists(PROJETO):
    !git clone -b {BRANCH} https://github.com/elbergaliza/Tech-Challenge-FIAP-FASE4.git {PROJETO}
else:
    !cd {PROJETO} && git fetch origin {BRANCH} && git checkout {BRANCH} && git pull origin {BRANCH}

!pip install -q -r {PROJETO}/requirements.txt

# No Colab há conflito mediapipe x tensorflow pré-instalado
!pip uninstall -y -q tensorflow tensorflow-cpu 2>/dev/null

%cd {PROJETO}
print('Pronto!')

## 2. Baixar dados reais do eICU Demo (PhysioNet)

O eICU Collaborative Research Database Demo é público e não exige login para
os arquivos de demo.

> **Dica de velocidade:** o download pode ser lento. Se quiser pular, use
> os dados mockados na célula 3.

In [ ]:
import os
import urllib.request

EICU_DIR = f'{PROJETO}/eicu_anomaly_detection/data/raw'
os.makedirs(EICU_DIR, exist_ok=True)

base_url = 'https://physionet.org/files/eicu-crd-demo/2.0.1/'
arquivos = ['vitalPeriodic.csv.gz', 'lab.csv.gz', 'medication.csv.gz']

for arq in arquivos:
    destino = os.path.join(EICU_DIR, arq)
    if os.path.exists(destino):
        print(f'{arq} ja existe.')
        continue
    url = base_url + arq
    print(f'Baixando {arq}...')
    try:
        urllib.request.urlretrieve(url, destino)
        print(f'  OK: {destino} ({os.path.getsize(destino) / 1024 / 1024:.1f} MB)')
    except Exception as e:
        print(f'  ERRO: {e}')
        raise

print('Dados eICU prontos.')

## 3. Definir o vídeo para o módulo de vídeo

O notebook usa a variável de ambiente `REHAB_VIDEO_ID` para escolher
qual vídeo do dataset REHAB24-6 será analisado.

- Valor padrão: `PM_029` (vídeo já incluso no repositório).
- Para usar outro vídeo, defina `REHAB_VIDEO_ID` antes de executar a célula
  abaixo (ex.: `PM_038`).
- IDs disponíveis para Camera17 no Ex6: PM_008, PM_022, PM_029, PM_038,
  PM_043, PM_105, PM_113, PM_118, PM_126.

```python
import os
os.environ['REHAB_VIDEO_ID'] = 'PM_029'  # altere aqui se desejar
```

In [ ]:
import os

# Le a variavel de ambiente REHAB_VIDEO_ID.
# Se nao estiver definida, usa 'PM_029' (video padrao do repositorio).
VIDEO_ID = os.environ.get('REHAB_VIDEO_ID', 'PM_029')

VIDEO_REAL = f'{PROJETO}/modulo_video/data/exemplos/{VIDEO_ID}-Camera17-30fps.mp4'
VIDEO_PATIENT_ID = VIDEO_ID
print(f'Video selecionado: {VIDEO_REAL}')
print(f'Existe: {os.path.exists(VIDEO_REAL)}')

### Opção avançada — baixar outro vídeo do REHAB24-6

Se `REHAB_VIDEO_ID` aponta para um vídeo que não está em
`modulo_video/data/exemplos/`, execute a célula abaixo para baixar apenas
esse vídeo do dataset REHAB24-6 (Zenodo). O download do arquivo compactado
é de ~2.6 GB, então pode demorar alguns minutos. Se o vídeo já existir,
a célula pula automaticamente.

In [ ]:
import os
import urllib.request
import zipfile

VIDEO_ID = os.environ.get('REHAB_VIDEO_ID', 'PM_029')

VIDEO_LOCAL = f'{PROJETO}/modulo_video/data/exemplos/{VIDEO_ID}-Camera17-30fps.mp4'
if os.path.exists(VIDEO_LOCAL):
    print(f'Video {VIDEO_ID} ja existe no repositorio. Nenhum download necessario.')
    VIDEO_REAL = VIDEO_LOCAL
    VIDEO_PATIENT_ID = VIDEO_ID
else:
    REHAB_DIR = '/content/REHAB24-6'
    os.makedirs(REHAB_DIR, exist_ok=True)

    videos_zip = '/tmp/rehab_videos.zip'
    video_rehab = f'{REHAB_DIR}/videos/Ex6/{VIDEO_ID}-Camera17-30fps.mp4'
    if not os.path.exists(video_rehab):
        print(f'Baixando videos.zip do REHAB24-6 (2.6 GB, pode levar)...')
        urllib.request.urlretrieve('https://zenodo.org/records/13305826/files/videos.zip', videos_zip)
        print(f'Descompactando apenas o video {VIDEO_ID}...')
        with zipfile.ZipFile(videos_zip) as z:
            alvo = f'Ex6/{VIDEO_ID}-Camera17-30fps.mp4'
            z.extract(alvo, f'{REHAB_DIR}/videos')
        print('OK')

    VIDEO_REAL = video_rehab
    VIDEO_PATIENT_ID = VIDEO_ID

print(f'Video selecionado: {VIDEO_REAL}')
print(f'Existe: {os.path.exists(VIDEO_REAL)}')

## 4. Verificar dados antes da fusão

In [ ]:
import os
print(f'VIDEO_REAL: {VIDEO_REAL}')
print(f'VIDEO_REAL existe: {os.path.exists(VIDEO_REAL)}')
print(f'EICU_DIR: {EICU_DIR}')
print(f'EICU_DIR existe: {os.path.exists(EICU_DIR)}')
for arq in ['vitalPeriodic.csv.gz', 'lab.csv.gz', 'medication.csv.gz']:
    print(f'  {arq}: {os.path.exists(os.path.join(EICU_DIR, arq))}')

## 4b. (Opcional) Módulo de áudio/texto

O módulo de áudio extrai features acústicas e transcreve a fala usando **Azure Cognitive Services**.
É opcional: se nenhum arquivo for fornecido, a fusão roda normalmente com clínico + vídeo.

**Para ativar:**
1. Faça upload de um arquivo `.wav` ou `.mp3` na célula abaixo, **ou** informe o caminho de um arquivo já presente no Colab.
2. Adicione suas credenciais Azure como **Secrets do Colab** (ícone 🔑 na barra lateral):
   - `AZURE_SPEECH_KEY`
   - `AZURE_SPEECH_REGION`
   - `AZURE_TEXT_ANALYTICS_ENDPOINT`
   - `AZURE_TEXT_ANALYTICS_KEY`
3. Execute a célula — ela exporta as credenciais como variáveis de ambiente.

> Se as credenciais não estiverem presentes, o pipeline usa fallback acústico (sem transcrição).


In [ ]:
import os

# ── Credenciais Azure (via Colab Secrets ou variáveis de ambiente) ────────────
try:
    from google.colab import userdata
    for key in [
        'AZURE_SPEECH_KEY', 'AZURE_SPEECH_REGION',
        'AZURE_TEXT_ANALYTICS_ENDPOINT', 'AZURE_TEXT_ANALYTICS_KEY',
    ]:
        val = userdata.get(key)
        if val:
            os.environ[key] = val
    print('Credenciais Azure carregadas dos Secrets do Colab.')
except Exception:
    print('[aviso] Sem acesso a Colab Secrets — usando variáveis de ambiente existentes.')

# ── Arquivo de áudio ──────────────────────────────────────────────────────────
# Opção A: upload interativo (mude para True para fazer upload de um arquivo local)
UPLOAD_AUDIO = False

AUDIO_PATH = os.environ.get('AUDIO_PATH', '').strip()  # pode definir via env também

if UPLOAD_AUDIO and not AUDIO_PATH:
    from google.colab import files
    print('Selecione um arquivo .wav ou .mp3:')
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        AUDIO_PATH = f'/content/{fname}'
        print(f'Arquivo carregado: {AUDIO_PATH}')
    else:
        print('Nenhum arquivo carregado.')

# Opção B: arquivo já presente no Colab (descomente e edite o caminho se desejar)
# AUDIO_PATH = '/content/minha_gravacao.wav'

# Opção C: fallback automático — detecta qualquer .wav/.mp3 recente em /content
# (útil quando o upload é feito pelo widget do Colab e a variável não foi setada)
if not AUDIO_PATH or not os.path.exists(AUDIO_PATH):
    import glob
    candidatos = sorted(
        glob.glob('/content/*.wav') + glob.glob('/content/*.mp3'),
        key=os.path.getmtime,
        reverse=True,
    )
    if candidatos:
        AUDIO_PATH = candidatos[0]
        print(f'Áudio detectado automaticamente: {AUDIO_PATH}')

# ── Resultado ─────────────────────────────────────────────────────────────────
if AUDIO_PATH and os.path.exists(AUDIO_PATH):
    AUDIO_FLAG = f'--audio {AUDIO_PATH}'
    print(f'Áudio configurado: {AUDIO_PATH}')
else:
    AUDIO_FLAG = ''
    print('Módulo de áudio desativado (nenhum arquivo fornecido).')


## 4c. (Opcional) Preparar áudio do dataset médico deepgram-diarized-272

Dataset com 272 consultas médicas simuladas (inglês), diarizado por speaker.
A célula abaixo:
1. Baixa **1 arquivo MP3** do dataset (sem baixar os 23 GB completos)
2. Lê o JSON de diarização para identificar os segmentos **do paciente**
3. Concatena e exporta um WAV com apenas a fala do paciente
4. Configura `AUDIO_FLAG` e `AUDIO_LANGUAGE` para processar em `en-US`

> **Pré-requisito:** `huggingface_hub` (já incluído nos requirements).
> O corte usa `pydub` + `ffmpeg` (disponíveis no Colab por padrão).


In [ ]:
import os
import json
from pathlib import Path
from pydub import AudioSegment
from huggingface_hub import hf_hub_download

DATASET_ID  = "wmatbooth/medical-conversation-deepgram-diarized-272"
# Escolha qualquer arquivo RES0001..RES0213 (resp), MSK0001..MSK0046 (músculo-esquelético)
AUDIO_FILE  = os.environ.get("DEEPGRAM_AUDIO_FILE", "RES0001")
AUDIO_DIR   = f"{PROJETO}/data/audio/deepgram"
os.makedirs(AUDIO_DIR, exist_ok=True)

print(f"Baixando {AUDIO_FILE}.mp3 ...")
mp3_path = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename=f"audio/{AUDIO_FILE}.mp3",
    local_dir=AUDIO_DIR,
)
print(f"  → {mp3_path}")

print(f"Baixando diarização {AUDIO_FILE}.json ...")
conv_json_path = hf_hub_download(
    repo_id=DATASET_ID,
    repo_type="dataset",
    filename=f"deepgram/conversation_json/{AUDIO_FILE}.json",
    local_dir=AUDIO_DIR,
)
print(f"  → {conv_json_path}")

# ── Identificar speaker do paciente ──────────────────────────────────────────
# O Deepgram numera speakers 0,1,2... O paciente é geralmente o speaker
# com MENOS tempo de fala total (médico fala mais). Ajuste se necessário.
with open(conv_json_path) as f:
    raw = json.load(f)

# suporta lista plana ou {"turns": [...]}
turns = raw["turns"] if isinstance(raw, dict) else raw   # lista de {speaker, start, end, transcript}

speaker_duration: dict[int, float] = {}
for turn in turns:
    spk = turn.get("speaker", 0)
    dur = turn.get("end", 0) - turn.get("start", 0)
    speaker_duration[spk] = speaker_duration.get(spk, 0) + dur

patient_speaker = min(speaker_duration, key=speaker_duration.get)
print(f"\nSpeakers detectados: {speaker_duration}")
print(f"Speaker identificado como PACIENTE: {patient_speaker}")

patient_turns = [t for t in turns if t.get("speaker") == patient_speaker]
print(f"Segmentos do paciente: {len(patient_turns)}")
for t in patient_turns[:3]:
    print(f"  [{t['start']:.1f}s → {t['end']:.1f}s] {t.get('transcript','')[:60]}")
print("  ...")

# ── Cortar e concatenar segmentos do paciente ─────────────────────────────────
print("\nCarregando MP3 completo (pode demorar ~30s)...")
full_audio = AudioSegment.from_mp3(mp3_path)

patient_audio = AudioSegment.empty()
for turn in patient_turns:
    start_ms = int(turn["start"] * 1000)
    end_ms   = int(turn["end"]   * 1000)
    patient_audio += full_audio[start_ms:end_ms]

duration_s = len(patient_audio) / 1000
print(f"Duração total da fala do paciente: {duration_s:.1f}s")

# ── Exportar WAV ──────────────────────────────────────────────────────────────
patient_wav = f"{AUDIO_DIR}/{AUDIO_FILE}_patient_only.wav"
patient_audio.export(patient_wav, format="wav")
print(f"WAV exportado: {patient_wav}")

# ── Configurar pipeline para en-US ────────────────────────────────────────────
AUDIO_FLAG     = f"--audio {patient_wav} --audio-language en-US"
AUDIO_LANGUAGE = "en-US"
print(f"\nAUDIO_FLAG configurado: {AUDIO_FLAG}")
print("Pronto para processar em en-US com termos críticos em inglês.")


## 5. Executar a fusão multimodal

Esta é a célula principal. Ela roda `main.py`, que orquestra o módulo clínico,
o módulo de vídeo e a fusão dos alertas em um único relatório JSON.

In [ ]:
import os
os.makedirs(f'{PROJETO}/outputs', exist_ok=True)

# AUDIO_LANGUAGE pode ser sobrescrito pela célula 4c (deepgram → en-US)
audio_lang = locals().get('AUDIO_LANGUAGE', 'pt-BR')

# Evita passar --audio-language duas vezes quando AUDIO_FLAG já o contém
if '--audio-language' in AUDIO_FLAG:
    audio_lang_flag = ''
else:
    audio_lang_flag = f'--audio-language {audio_lang}'

!python main.py \
  --video {VIDEO_REAL} \
  --eicu-data {EICU_DIR} \
  --clinical-patient-id 143870 \
  --video-patient-id {VIDEO_PATIENT_ID} \
  {AUDIO_FLAG} \
  {audio_lang_flag} \
  --sem-objetos \
  --saida {PROJETO}/outputs/final_multimodal_report.json


## 6. Visualizar relatório final e resumo por módulo

In [ ]:
import json
with open(f'{PROJETO}/outputs/final_multimodal_report.json') as f:
    relatorio = json.load(f)

resumo = relatorio.get('resumo', {})
alertas = relatorio.get('alertas', [])

print('=' * 60)
print('RESUMO FINAL')
print('=' * 60)
print(f'  Score médio:         {resumo.get("score_medio", "--")}')
print(f'  Nível mais crítico:  {resumo.get("nivel_mais_critico", "--").upper()}')
print(f'  Total de alertas:    {resumo.get("total_alertas", 0)}')
print(f'  Recomendação:        {resumo.get("recomendacao_geral", "--")}')
print()

print('ALERTAS POR MÓDULO:')
por_modulo = {}
for alerta in alertas:
    modulo = alerta.get('modulo')
    por_modulo.setdefault(modulo, []).append(alerta)

for modulo, lista in sorted(por_modulo.items()):
    scores = [a['score_risco'] for a in lista]
    score_max = max(scores)
    print(f'  {modulo}: {len(lista)} alerta(s) | score max = {score_max:.3f}')
    for a in lista:
        print(f'    • [{a["nivel_risco"].upper():8s}] {a["descricao"][:80]}')

print()
print('RELATÓRIO COMPLETO:')
print(json.dumps(relatorio, ensure_ascii=False, indent=2))


## 7. (opcional) Baixar resultados

In [ ]:
from google.colab import files
files.download(f'{PROJETO}/outputs/final_multimodal_report.json')

---

## 8. (opcional) Calibração com REHAB24-6

Esta seção é opcional. Ela baixa o dataset completo do Zenodo, calibra
limiares para agachamento (Ex6) e mostra a comparação antes/depois.
Use `--max 4` para teste rápido, ou remova para processar todas as gravações.

In [ ]:
import urllib.request
import zipfile

REHAB_DIR = '/content/REHAB24-6'
os.makedirs(REHAB_DIR, exist_ok=True)

if not os.path.exists(f'{REHAB_DIR}/videos/Ex6'):
    print('Baixando videos.zip (2.6 GB)...')
    urllib.request.urlretrieve('https://zenodo.org/records/13305826/files/videos.zip', '/tmp/rehab_videos.zip')
    print('Descompactando...')
    with zipfile.ZipFile('/tmp/rehab_videos.zip') as z:
        z.extractall(f'{REHAB_DIR}/videos')
    print('OK')

if not os.path.exists(f'{REHAB_DIR}/Segmentation.csv'):
    print('Baixando Segmentation.csv...')
    urllib.request.urlretrieve('https://zenodo.org/records/13305826/files/Segmentation.csv', f'{REHAB_DIR}/Segmentation.csv')
    print('OK')

print('Dataset pronto.')

In [ ]:
# Calibrar com apenas 4 gravações (rápido) ou remover --max 4 para todas
!python -m modulo_video.calibrar --raiz {REHAB_DIR} --camera Camera17 --exercise 6 --pular-frames 5 --max 4 --salvar-limiares modulo_video/data/saida/limiares_calibrados.json

In [ ]:
with open(f'{PROJETO}/modulo_video/data/saida/limiares_calibrados.json') as f:
    print(json.dumps(json.load(f), ensure_ascii=False, indent=2))

In [ ]:
# Comparar alerta com limiares calibrados
!python -m modulo_video.comparar_video --video {VIDEO_REAL} --limiares modulo_video/data/saida/limiares_calibrados.json --sem-objetos